# 特征分箱统计分析 - feature_bin_stats

本 Notebook 演示 `feature_bin_stats` 函数的完整功能，包括：
- 基础分箱分析
- 金额口径分析
- Rule规则分析
- 规则集分析

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from hscredit.report import feature_bin_stats
from hscredit.core.rules import Rule
from hscredit.report.ruleset_report import ruleset_report

## 1. 加载数据

In [2]:
# 加载百融数据
data = pd.read_excel('hscredit.xlsx')
print(f"数据形状: {data.shape}")
data.head()

数据形状: (22729, 10)


,MOB1,MOB2,青云24,游昆定制分80,百融定制分V9,中智小牛分C3,度小满欺诈因子V6PRO多头版,百行百川分FPV1,设备黑名单,放款期数
0,0,0,612.0,718.0,914.739990,687.0,NaN,NaN,2.0,6
1,0,0,640.0,718.0,894.280029,774.0,NaN,NaN,3.0,6
2,0,0,581.0,709.0,822.119995,629.0,53.380001,306.0,3.0,6
3,0,0,650.0,718.0,794.500000,662.0,48.650002,285.0,3.0,6
4,0,0,650.0,718.0,794.500000,662.0,48.650002,285.0,3.0,6


## 2. 基础分箱分析

In [3]:
# 创建目标变量
data['target'] = (data['MOB1'] > 15).astype(int)

# 基础分箱分析
table = feature_bin_stats(
    data,
    '百融定制分V9',
    target='target',
    method='optimal',
    max_n_bins=5,
    desc='百融定制分V9'
)
table

,指标名称,指标含义,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,累积LIFT值,累积坏账改善,累积好样本数,累积坏样本数,分档KS值
0,百融定制分V9,百融定制分V9,"(-inf, 665.8874]",3840,3404,436,0.168947,0.161159,0.271313,0.113542,0.520882,0.057377,0.1892,1.6059,0.1232,1.6059,0.1232,3404,436,0.110154
1,百融定制分V9,百融定制分V9,"(665.8874, 785.7469]",9507,8747,760,0.418276,0.414118,0.472931,0.079941,0.132798,0.007810,0.1892,1.1307,0.0940,1.2674,0.3804,12151,1196,0.168967
2,百融定制分V9,百融定制分V9,"(785.7469, 847.2135]",5597,5304,293,0.246249,0.251113,0.182327,0.052349,-0.320098,0.022018,0.1892,0.7404,-0.0848,1.1117,0.5591,17455,1489,0.100182
3,百融定制分V9,百融定制分V9,"(847.2135, 893.2583]",2729,2629,100,0.120067,0.124467,0.062228,0.036643,-0.693243,0.043147,0.1892,0.5183,-0.0657,1.0370,0.7589,20084,1589,0.037942
4,百融定制分V9,百融定制分V9,"(893.2583, +inf)",1053,1035,18,0.046328,0.049001,0.011201,0.017094,-1.475839,0.055787,0.1892,0.2418,-0.0368,1.0001,1.0000,21119,1607,0.000142
5,百融定制分V9,百融定制分V9,缺失,3,3,0,0.000132,0.000142,0.000000,0.000000,-21.548517,0.003061,0.1892,0.0000,-0.0001,1.0000,0.0000,21122,1607,0.000000


## 3. 不同分箱方法对比

In [4]:
methods = ['optimal', 'cart', 'quantile']

for method in methods:
    print(f"\n=== {method} ===")
    table = feature_bin_stats(
        data,
        '百融定制分V9',
        target='target',
        method=method,
        max_n_bins=5
    )
    print(f"IV值: {table['指标IV值'].iloc[0]:.4f}")
    print(f"分箱数: {len(table)}")


=== optimal ===
IV值: 0.1892
分箱数: 6

=== cart ===
IV值: 0.1961
分箱数: 6

=== quantile ===
IV值: 0.1696
分箱数: 6


## 4. 多特征分析

In [14]:
features = ['百融定制分V9', '青云24', '游昆定制分80']

tables = []
for feat in features:
    table = feature_bin_stats(
        data,
        feat,
        target='target',
        method='optimal',
        max_n_bins=5,
        desc=feat
    )
    tables.append(table)

multi_table = pd.concat(tables, axis=0, ignore_index=True)
multi_table[['指标名称', '分箱标签', '样本总数', '坏样本率', '指标IV值']].head(10)

,指标名称,分箱标签,样本总数,坏样本率,指标IV值
0,百融定制分V9,"(-inf, 665.8874]",3840,0.113542,0.18920
1,百融定制分V9,"(665.8874, 785.7469]",9507,0.079941,0.18920
2,百融定制分V9,"(785.7469, 847.2135]",5597,0.052349,0.18920
3,百融定制分V9,"(847.2135, 893.2583]",2729,0.036643,0.18920
4,百融定制分V9,"(893.2583, +inf)",1053,0.017094,0.18920
5,百融定制分V9,缺失,3,0.000000,0.18920
6,青云24,"(-inf, 552.69]",5535,0.089792,0.06678
7,青云24,"(552.69, 617.67]",8165,0.074954,0.06678
8,青云24,"(617.67, 693.48]",7172,0.060234,0.06678
9,青云24,"(693.48, 773.9]",1732,0.037529,0.06678


## 5. 金额口径分析

使用 `amount` 参数进行金额口径分析。分箱表列名与样本口径统一，便于统一处理。

In [15]:
# 创建示例数据（带金额字段）
np.random.seed(42)
n_samples = 1000

demo_data = pd.DataFrame({
    'score': np.random.randint(300, 800, n_samples),
    'target': np.random.binomial(1, 0.1, n_samples),
    'loan_amount': np.random.randint(10000, 100000, n_samples),
})

print(f"数据集样本数: {len(demo_data)}")
demo_data.head()

数据集样本数: 1000


,score,target,loan_amount
0,402,0,18906
1,735,0,46914
2,648,0,55379
3,570,1,88612
4,406,0,47650


### 5.1 样本数口径 vs 金额口径

In [16]:
# 样本数口径
print('=== 样本数口径 ===')
table_sample = feature_bin_stats(
    data=demo_data,
    feature='score',
    target='target',
    method='quantile',
    max_n_bins=5,
    desc='信用评分'
)
print(table_sample[['分箱标签', '样本总数', '好样本数', '坏样本数', '坏样本率']].head())

=== 样本数口径 ===
             分箱标签  样本总数  好样本数  坏样本数      坏样本率
0   (-inf, 411.8]   200   184    16  0.080000
1  (411.8, 497.0]   201   179    22  0.109453
2  (497.0, 595.8]   199   182    17  0.085427
3  (595.8, 696.0]   201   185    16  0.079602
4   (696.0, +inf)   199   175    24  0.120603


In [17]:
# 金额口径
print('\n=== 金额口径 ===')
table_amount = feature_bin_stats(
    data=demo_data,
    feature='score',
    target='target',
    method='quantile',
    max_n_bins=5,
    desc='信用评分',
    amount='loan_amount'
)
print(table_amount[['分箱标签', '样本总数', '好样本数', '坏样本数', '坏样本率']].head())

print(f"\n列名是否一致: {list(table_amount.columns) == list(table_sample.columns)}")


=== 金额口径 ===
             分箱标签        样本总数        好样本数       坏样本数      坏样本率
0   (-inf, 411.8]  10682751.0   9950776.0   731975.0  0.068519
1  (411.8, 497.0]  10779435.0   9430052.0  1349383.0  0.125181
2  (497.0, 595.8]  10629973.0   9632461.0   997512.0  0.093840
3  (595.8, 696.0]  10862911.0  10037484.0   825427.0  0.075986
4   (696.0, +inf)  10768532.0   9434814.0  1333718.0  0.123853

列名是否一致: True


## 8. 总结

**主要功能**：
- `feature_bin_stats`: 特征分箱统计分析
- `Rule.report`: 单规则分析
- `ruleset_report`: 规则集分析

**金额口径特点**：
- 使用 `amount` 参数指定金额字段
- 分箱表列名与样本口径统一
- 金额口径下，样本总数等列存储的是金额汇总数据